# Lab 2 — Baseline com múltiplos datasets

Neste notebook vamos rodar o mesmo pipeline de baseline para 4 datasets diferentes.

A ideia é simples: para cada dataset na lista `keys_to_run`, o código vai:
1. baixar e carregar os dados
2. limpar valores ausentes e aplicar `dropna` antes do split
3. treinar dois modelos baseline (um linear e um de floresta)
4. imprimir as métricas e salvar os resultados

No final, uma tabela compara os melhores modelos de cada dataset.

In [1]:
import os

base = r"C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga"

for root, dirs, files in os.walk(base):
    print(root)
    for d in dirs:
        print("  [DIR]", d)
    for f in files:
        print("  [FILE]", f)

C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga
  [DIR] corruptions
  [DIR] evaluation
  [DIR] tasks
  [DIR] __pycache__
  [FILE] basis.py
  [FILE] utils.py
  [FILE] __init__.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\corruptions
  [DIR] __pycache__
  [FILE] generic.py
  [FILE] image.py
  [FILE] numerical.py
  [FILE] text.py
  [FILE] __init__.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\corruptions\__pycache__
  [FILE] generic.cpython-39.pyc
  [FILE] __init__.cpython-39.pyc
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\evaluation
  [FILE] basis.py
  [FILE] corruption_impact.py
  [FILE] schema_stresstest.py
  [FILE] schema_validation.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\tasks
  [FILE] income.py
  [FILE] openml.py
  [FILE] reviews.py
  [FILE] shoes.py
C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\__pycache__
  [FILE] basis.cpython-39.pyc
  [FILE] utils.cpython-39.pyc
  [FILE] __init__.cpytho

In [2]:
# 1) Bibliotecas
from __future__ import annotations

import json
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

import sys
from pathlib import Path

import sys
from pathlib import Path

BASE_DIR = Path().resolve()
JENGA_PATH = BASE_DIR / "jenga" / "src"

sys.path.insert(0, str(JENGA_PATH))

import jenga
print("Jenga path:", jenga.__file__)

from jenga.corruptions.generic import MissingValues
print("IMPORT OK")

from jenga.corruptions.generic import MissingValues

import inspect
print(inspect.signature(MissingValues))

from joblib import Parallel, delayed

Jenga path: C:\Users\kslima\mestrado\cienciaDados\atv2\jenga\src\jenga\__init__.py
IMPORT OK
(column, fraction, na_value=nan, missingness='MCAR')


In [3]:
# 2) Metadados dos datasets e funcoes  que nos ajudam a baixar, extrair e normalizar os dados

DATASETS = {
    "adult": {"task": "classification", "target": "income"},
    "bank_marketing": {"task": "classification", "target": "y"},
    "air_quality_uci": {"task": "regression", "target": "C6H6(GT)"},
    "communities_crime": {"task": "regression", "target": "ViolentCrimesPerPop"},
}

# Tokens que representam valores ausentes
NUMERIC_MISSING_VALUES = [-200, -200.0]

STRING_MISSING_VALUES = [
    "?", " ?", "? ", "NA", "N/A", "na", "n/a", "NaN", "nan", "", " ",
    "unknown", "Unknown", "-200",
]


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def download_file(url: str, output_path: Path) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not output_path.exists():
        print(f"[download] {url}")
        urlretrieve(url, output_path)
    else:
        print(f"[cache] {output_path}")
    return output_path


def unzip_file(zip_path: Path, extract_dir: Path) -> Path:
    extract_dir.mkdir(parents=True, exist_ok=True)
    marker = extract_dir / ".extracted"
    if not marker.exists():
        print(f"[extract] {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)
        marker.write_text("ok\n", encoding="utf-8")
    else:
        print(f"[cache] {extract_dir}")
    return extract_dir


def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Substitui tokens de missing por NaN, coluna a coluna, respeitando o dtype.
    
    - Colunas numericas: substitui -200 e -200.0
    - Colunas de texto: faz strip e substitui tokens como '?', 'unknown', etc.
    Essa abordagem evita erros do pandas 2.x ao misturar int e str no replace().
    -- TODO: verificar se tem outros simbolos estranhos nos dataset da atividade (pedir par aos alunos checarem isso)
    """
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].replace(NUMERIC_MISSING_VALUES, np.nan)
        else:
            df[col] = df[col].astype("string").str.strip()
            df[col] = df[col].replace(STRING_MISSING_VALUES, pd.NA)
    return df
def to_native_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Converte TODAS as colunas com dtypes de extensão do pandas para tipos nativos,
    eliminando pd.NA que causa 'boolean value of NA is ambiguous'.
    Cobre: StringDtype, BooleanDtype, Int8/16/32/64Dtype, Float32/64Dtype, etc.
    """
    df = df.copy()
    for col in df.columns:
        dtype = df[col].dtype
        # Detecta qualquer ExtensionDtype (StringDtype, BooleanDtype, Int64Dtype, etc.)
        if not isinstance(dtype, np.dtype):
            if hasattr(dtype, "numpy_dtype"):
                # Tipos numéricos nullable (Int64, Float64) → float64 nativo
                # (float64 aceita np.nan; int não aceita, por isso usa float)
                try:
                    df[col] = df[col].astype(object).where(
                        df[col].notna(), other=np.nan
                    ).astype(float)
                    continue
                except (ValueError, TypeError):
                    pass
            # Todos os outros (StringDtype, BooleanDtype, CategoricalDtype, etc.) → object
            df[col] = df[col].astype(object).where(df[col].notna(), other=None)
    return df

def _to_object_dtype(X):
    if hasattr(X, "astype"):
        return X.astype(object)
    return X

def _to_str_dtype(X):
    return X.astype(str)

In [4]:
# 3) Funcoes. para baixar os datasets automaticamente

# fiz alguns pequenos ajustes caso a caso para facilitar a leitura e evitar erros de parsing

def load_adult(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "adult"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
        base / "adult.data",
    )
    test_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
        base / "adult.test",
    )

    columns = [
        "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income",
    ]

    train_df = pd.read_csv(data_file, header=None, names=columns, skipinitialspace=True, na_values=["?", " ?"])
    test_df = pd.read_csv(test_file, header=None, names=columns, skiprows=1, skipinitialspace=True, na_values=["?", " ?"])

    df = pd.concat([train_df, test_df], ignore_index=True)
    df["income"] = df["income"].astype(str).str.replace(".", "", regex=False).str.strip()
    return df, "income", "classification"


def load_bank_marketing(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "bank_marketing"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip",
        base / "bank.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    df = pd.read_csv(extract_dir / "bank-full.csv", sep=";")
    return df, "y", "classification"


def load_air_quality_uci(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "air_quality_uci"
    zip_path = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip",
        base / "AirQualityUCI.zip",
    )
    extract_dir = unzip_file(zip_path, base / "extracted")
    candidates = list(extract_dir.rglob("AirQualityUCI.csv"))
    if not candidates:
        raise FileNotFoundError("Nao encontrei AirQualityUCI.csv dentro do zip.")

    df = pd.read_csv(candidates[0], sep=";", decimal=",")
    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    for col in ["Date", "Time"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    return df, "C6H6(GT)", "regression"


def load_communities_crime(data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    base = data_dir / "communities_crime"
    data_file = download_file(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/communities/communities.data",
        base / "communities.data",
    )
    df = pd.read_csv(data_file, header=None, na_values=["?"])

    cols = [f"col_{i}" for i in range(df.shape[1])]
    cols[-1] = "ViolentCrimesPerPop"
    df.columns = cols
    # colunas  com muito problema
    df = df.drop(columns=[c for c in ["col_0", "col_1", "col_2", "col_3", "col_4"] if c in df.columns])
    return df, "ViolentCrimesPerPop", "regression"


def load_dataset(key: str, data_dir: Path) -> tuple[pd.DataFrame, str, str]:
    if key == "adult":
        return load_adult(data_dir)
    if key == "bank_marketing":
        return load_bank_marketing(data_dir)
    if key == "air_quality_uci":
        return load_air_quality_uci(data_dir)
    if key == "communities_crime":
        return load_communities_crime(data_dir)
    raise ValueError(f"Key desconhecida: {key}")

def inject_mcar(df: pd.DataFrame, missing_rate: float, random_state: int = 42) -> pd.DataFrame:
    corruption = MissingValues(missing_rate=missing_rate, random_state=random_state)
    df_corrupted = corruption.transform(df.copy())
    return df_corrupted


def inject_mar(df: pd.DataFrame, missing_rate: float, random_state: int = 42) -> pd.DataFrame:
    df_corrupted = df.copy()

    np.random.seed(random_state)

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) == 0:
        return inject_mcar(df, missing_rate, random_state)

    ref_col = numeric_cols[0]

    threshold = df[ref_col].median()

    mask = df[ref_col] > threshold

    for col in df.columns:
        prob = np.random.rand(len(df))
        df_corrupted.loc[mask & (prob < missing_rate), col] = np.nan

    return df_corrupted


def inject_mnar(df: pd.DataFrame, missing_rate: float, random_state: int = 42) -> pd.DataFrame:
    df_corrupted = df.copy()

    np.random.seed(random_state)

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) == 0:
        return inject_mcar(df, missing_rate, random_state)

    for col in numeric_cols:
        threshold = df[col].median()
        mask = df[col] > threshold

        prob = np.random.rand(len(df))
        df_corrupted.loc[mask & (prob < missing_rate), col] = np.nan

    return df_corrupted

In [5]:
# 4) Funcoes para separar treino e teste

FAST_MODE          = False
MISSING_MECHANISMS = ["MCAR", "MAR", "MNAR"]
MISSING_RATES      = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
N_ESTIMATORS       = 50
MAX_ITER_IMPUTER   = 5
SAMPLE_SIZE        = None

def split_X_y(df: pd.DataFrame, target: str) -> tuple[pd.DataFrame, pd.Series]:
    if target not in df.columns:
        raise ValueError(f"Target '{target}' nao esta no dataset.")
    return df.drop(columns=[target]), df[target]


def build_preprocessor(X: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler() if scale_numeric else "passthrough"),
    ])

    cat_pipeline = Pipeline([
        ("to_object", FunctionTransformer(_to_object_dtype)),  # função nomeada
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("to_str", FunctionTransformer(_to_str_dtype)),        # função nomeada
        ("onehot", make_one_hot_encoder()),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", num_pipeline, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", cat_pipeline, categorical_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_preprocessor_with_imputer(
    X: pd.DataFrame,
    imputer_type: str,
    scale_numeric: bool
) -> ColumnTransformer:
    """
    imputer_type aceito:
      "dropna"        – sem imputação (linhas com NaN já foram removidas antes)
      "simple_mean"   – SimpleImputer(strategy="mean")
      "simple_median" – SimpleImputer(strategy="median")
      "simple_freq"   – SimpleImputer(strategy="most_frequent")
      "simple_const"  – SimpleImputer(strategy="constant", fill_value=0)
      "knn_3"         – KNNImputer(n_neighbors=3)
      "knn_5"         – KNNImputer(n_neighbors=5)
      "knn_7"         – KNNImputer(n_neighbors=7)
      "iterative"     – IterativeImputer
    """
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    # ── imputador numérico ──────────────────────────────────────────────────
    if imputer_type == "dropna":
        # dados já vieram sem NaN — usa mean como passthrough seguro
        num_imputer = SimpleImputer(strategy="mean")
    elif imputer_type == "simple_mean":
        num_imputer = SimpleImputer(strategy="mean")
    elif imputer_type == "simple_median":
        num_imputer = SimpleImputer(strategy="median")
    elif imputer_type == "simple_freq":
        num_imputer = SimpleImputer(strategy="most_frequent")
    elif imputer_type == "simple_const":
        num_imputer = SimpleImputer(strategy="constant", fill_value=0)
    elif imputer_type.startswith("knn_"):
        k = int(imputer_type.split("_")[1])
        num_imputer = KNNImputer(n_neighbors=k)
    elif imputer_type == "iterative":
        num_imputer = IterativeImputer(
            max_iter=MAX_ITER_IMPUTER,
            initial_strategy="mean",
            random_state=0,
        )
    else:
        raise ValueError(f"Imputer desconhecido: {imputer_type}")

    num_pipeline = Pipeline([
        ("imputer", num_imputer),
        ("scaler", StandardScaler() if scale_numeric else "passthrough"),
    ])

    cat_pipeline = Pipeline([
        ("to_object", FunctionTransformer(_to_object_dtype)),
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("to_str", FunctionTransformer(_to_str_dtype)),
        ("onehot", make_one_hot_encoder()),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", num_pipeline, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", cat_pipeline, categorical_cols))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def inject_missing_values(df, target, mechanism, missing_rate, random_state):
    df_corrupted = df.copy()

    y = df_corrupted[target]
    X = df_corrupted.drop(columns=[target]).copy()

    for col in X.columns:
        corruption = MissingValues(
            column=col,              
            fraction=missing_rate,
            missingness=mechanism
        )

        X = corruption.transform(X)

    df_final = X.copy()
    df_final[target] = y

    return df_final


def run_classification(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:

    y = pd.Series(y.to_numpy(dtype=object, na_value=np.nan), index=y.index)
    y = y.dropna()
    y = y.astype(str)
    X = X.loc[y.index]
    X = to_native_dtypes(X)  # converte todos ExtensionDtypes para nativos

    class_counts = y.value_counts()
    stratify = y if int(class_counts.min()) >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify
    )

    models = {
        "logistic_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestClassifier(
                n_estimators=N_ESTIMATORS, 
                random_state=random_state,
                n_jobs=1,
                class_weight="balanced_subsample",
            )),
        ]),
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        metrics = {
            "accuracy": float(accuracy_score(y_test, pred)),
            "macro_f1": float(f1_score(y_test, pred, average="macro")),
            "weighted_f1": float(f1_score(y_test, pred, average="weighted")),
        }

        results[name] = metrics
        print(f"\n[model] {name}")
        print(metrics)

    return results

def run_classification_with_imputation(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float,
    random_state: int
) -> dict:

    # Lista completa de imputadores a comparar
    IMPUTERS_LIST = [
        "dropna",
        "simple_mean",
        "simple_median",
        "simple_freq",
        "simple_const",
        "knn_3",
        "knn_5",
        "knn_7",
        "iterative",
    ]

    y = pd.Series(y.to_numpy(dtype=object, na_value=np.nan), index=y.index)
    y = y.dropna()
    y = y.astype(str)
    X = X.loc[y.index].copy()
    X = to_native_dtypes(X)  # converte todos ExtensionDtypes para nativos

    if len(y) == 0:
        raise RuntimeError("Target vazia após limpeza.")

    class_counts = y.value_counts(dropna=False)
    stratify = y if int(class_counts.min()) >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=stratify
    )

    results = {}

    for imp in IMPUTERS_LIST:
        print(f"\n==== Imputador: {imp} ====")

        # dropna: remove linhas com NaN antes de treinar (sem imputação)
        if imp == "dropna":
            mask_train = ~X_train.isnull().any(axis=1)
            mask_test  = ~X_test.isnull().any(axis=1)
            X_tr = X_train[mask_train]
            y_tr = y_train[mask_train]
            X_te = X_test[mask_test]
            y_te = y_test[mask_test]
            if len(y_tr) == 0 or len(y_te) == 0:
                results[imp] = {}
                continue
        else:
            X_tr, y_tr, X_te, y_te = X_train, y_train, X_test, y_test

        models = {
            "logistic_regression": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, True)),
                ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
            ]),
            "random_forest": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, False)),
                ("model", RandomForestClassifier(
                    n_estimators=N_ESTIMATORS,
                    max_depth=10,
                    random_state=random_state,
                    n_jobs=1,
                    class_weight="balanced_subsample",
                )),
            ]),
        }

        results[imp] = {}

        for name, model in models.items():
            try:
                model.fit(X_tr, y_tr)
                pred = model.predict(X_te)

                metrics = {
                    "accuracy": float(accuracy_score(y_te, pred)),
                    "macro_f1": float(f1_score(y_te, pred, average="macro")),
                    "weighted_f1": float(f1_score(y_te, pred, average="weighted")),
                }

            except Exception as e:
                print(f"[ERRO] modelo={name}, imputer={imp}: {e}")
                metrics = {"accuracy": None, "macro_f1": None, "weighted_f1": None}

            results[imp][name] = metrics
            print(f"\n[model] {name}")
            print(metrics)

    return results

def run_regression(X: pd.DataFrame, y: pd.Series, test_size: float, random_state: int) -> dict:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    models = {
        "ridge_regression": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=True)),
            ("model", Ridge(alpha=1.0)),
        ]),
        "random_forest": Pipeline([
            ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
            ("model", RandomForestRegressor(
                n_estimators=N_ESTIMATORS, 
                max_depth=10,
                random_state=random_state,
                n_jobs=1
            )),
        ]),
    }

    results = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        mse = mean_squared_error(y_test, pred)
        metrics = {
            "mae": float(mean_absolute_error(y_test, pred)),
            "rmse": float(np.sqrt(mse)),
            "r2": float(r2_score(y_test, pred)),
        }

        results[name] = metrics
        print(f"\n[model] {name}")
        print(metrics)

    return results

def run_regression_with_imputation(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float,
    random_state: int
) -> dict:

    # Lista completa de imputadores a comparar
    IMPUTERS_LIST = [
        "dropna",
        "simple_mean",
        "simple_median",
        "simple_freq",
        "simple_const",
        "knn_3",
        "knn_5",
        "knn_7",
        "iterative",
    ]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    results = {}

    for imp in IMPUTERS_LIST:
        print(f"\n==== Imputador: {imp} ====")

        # dropna: remove linhas com NaN antes de treinar (sem imputação)
        if imp == "dropna":
            mask_train = ~X_train.isnull().any(axis=1)
            mask_test  = ~X_test.isnull().any(axis=1)
            X_tr = X_train[mask_train]
            y_tr = y_train[mask_train]
            X_te = X_test[mask_test]
            y_te = y_test[mask_test]
            if len(y_tr) == 0 or len(y_te) == 0:
                results[imp] = {}
                continue
        else:
            X_tr, y_tr, X_te, y_te = X_train, y_train, X_test, y_test

        models = {
            "ridge_regression": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, True)),
                ("model", Ridge(alpha=1.0)),
            ]),
            "random_forest": Pipeline([
                ("preprocess", build_preprocessor_with_imputer(X_tr, imp, False)),
                ("model", RandomForestRegressor(
                    n_estimators=N_ESTIMATORS, 
                    max_depth=10,
                    random_state=random_state,
                    n_jobs=1
                )),
            ]),
        }

        results[imp] = {}

        for name, model in models.items():
            model.fit(X_tr, y_tr)
            pred = model.predict(X_te)

            mse = mean_squared_error(y_te, pred)
            metrics = {
                "mae": float(mean_absolute_error(y_te, pred)),
                "rmse": float(np.sqrt(mse)),
                "r2": float(r2_score(y_te, pred)),
            }

            results[imp][name] = metrics
            print(f"\n[model] {name}")
            print(metrics)

    return results

# FUNÇÃO PRINCIPAL
import time
import sys
import json

def run_one_dataset(
    key: str,
    out_dir: str = "data",
    test_size: float = 0.2,
    random_state: int = 42
) -> dict:

    data_dir = Path(out_dir)
    df, target, task = load_dataset(key, data_dir=data_dir)
    
    if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
        df = df.sample(n=SAMPLE_SIZE, random_state=random_state).reset_index(drop=True)
        print(f"[FAST] Amostra de {SAMPLE_SIZE} linhas usada para {key}")

    print("\n" + "=" * 80)
    print(f"Dataset: {key} | Task: {task} | Target: {target}")
    print("=" * 80)

    df = normalize_missing_values(df)
    df = to_native_dtypes(df) 

    df[target] = df[target].astype(object)
    df = df[df[target].notna()].copy()

    if df.empty:
        raise RuntimeError("Dataset vazio após limpeza.")

    X, y = split_X_y(df, target)

    y = pd.Series([np.nan if v is pd.NA else v for v in y.astype(object)], index=y.index, dtype=object)

    if task == "classification":
        y = y.dropna()
        X = X.loc[y.index]

        X = to_native_dtypes(X)

        y = y.astype(object).astype(str)

        print("Distribuicao da target:")
        print(y.value_counts())

        baseline_results = run_classification(X, y, test_size, random_state)

    else:
        y = pd.to_numeric(y, errors="coerce")
        valid = y.notna()

        X = X.loc[valid]
        # NOVO: converte X também para dtypes nativos
        X = to_native_dtypes(X)
        y = y.loc[valid]

        baseline_results = run_regression(X, y, test_size, random_state)

    # EXPERIMENTOS
    missing_mechanisms = MISSING_MECHANISMS 
    missing_rates      = MISSING_RATES    

    all_results = []

    all_results.append({
        "dataset": key,
        "mechanism": "NONE",
        "rate": 0,
        "results": baseline_results
    })

    for mech in missing_mechanisms:
        for rate in missing_rates:
            print("\n" + "-" * 60)
            print(f"Rodando: {key}_{mech}_{int(rate*100)}")

            df_missing = inject_missing_values(
                df,
                target=target,
                mechanism=mech,
                missing_rate=rate,
                random_state=random_state
            )

            X, y = split_X_y(df_missing, target)
            X = to_native_dtypes(X)
            y = pd.Series(
                [np.nan if (v is pd.NA or (isinstance(v, float) and np.isnan(v))) else v 
                for v in y],
                index=y.index,
                dtype=object
            )

            if task == "classification":
                valid = y.notna()
                X = X.loc[valid]
                y = y.loc[valid].astype(str)
                res = run_classification_with_imputation(X, y, test_size, random_state)
            else:
                y = pd.to_numeric(y, errors="coerce")
                valid = y.notna()
                X = X.loc[valid]
                y = y.loc[valid]
                res = run_regression_with_imputation(X, y, test_size, random_state)

            all_results.append({
                "dataset": key,
                "mechanism": mech,
                "rate": rate,
                "results": res
            })

    return {
        "key": key,
        "task": task,
        "results": baseline_results,
        "experiments": all_results
    }

In [6]:
# 5) Configuracao da execucao

# ============================================================
# 🚀 FAST_MODE: True = rápido para testes  (~2-5 min)
#               False = completo para resultados finais (~120 min)
# ============================================================
FAST_MODE = True

# Parametros globais
out_dir = "data"
test_size = 0.2
random_state = 42

datasets_experiment = [
    "adult",
    "bank_marketing",
    "air_quality_uci",
    "communities_crime",
]

if FAST_MODE:
    MISSING_MECHANISMS = ["MCAR"]
    MISSING_RATES      = [0.05]
    N_ESTIMATORS       = 10
    MAX_ITER_IMPUTER   = 2
    SAMPLE_SIZE        = 3000
    print("⚡ FAST MODE — resultados aproximados para testes")
else:
    MISSING_MECHANISMS = ["MCAR", "MAR", "MNAR"]
    MISSING_RATES      = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
    N_ESTIMATORS       = 50
    MAX_ITER_IMPUTER   = 5
    SAMPLE_SIZE        = None
    print("🔬 FULL MODE — execução completa")

datasets_experiment

⚡ FAST MODE — resultados aproximados para testes


['adult', 'bank_marketing', 'air_quality_uci', 'communities_crime']

In [7]:
#barra de progresso
from tqdm import tqdm
from joblib import Parallel, delayed

class ParallelTqdm(Parallel):
    def __init__(self, total=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.total = total

    def __call__(self, *args, **kwargs):
        with tqdm(total=self.total) as self._pbar:
            return Parallel.__call__(self, *args, **kwargs)

    def print_progress(self):
        if self._pbar is not None:
            self._pbar.n = self.n_completed_tasks
            self._pbar.refresh()

In [8]:
from joblib import Parallel, delayed
import multiprocessing
from pathlib import Path
import pandas as pd

if __name__ == "__main__":

    # número de núcleos da máquina
    N_CORES = multiprocessing.cpu_count()

    # metade para paralelismo externo
    N_JOBS_OUTER = min(len(datasets_experiment), N_CORES // 2)

    print(f"Total cores: {N_CORES}")
    print(f"Paralelismo externo (datasets): {N_JOBS_OUTER}")


    # --- função segura ---
    def run_dataset_safe(key):
        try:
            return run_one_dataset(
                key=key,
                out_dir=out_dir,
                test_size=test_size,
                random_state=random_state,
            )
        except Exception as e:
            print(f"\n[ERRO] key={key}: {e}")
            return {
                "key": key,
                "task": DATASETS[key]["task"],
                "error": str(e)
            }


    #EXECUÇÃO PARALELA
    all_runs = ParallelTqdm(
        n_jobs=N_JOBS_OUTER,
        total=len(datasets_experiment),
        backend="threading",  # ← threads compartilham memória do notebook, sem problema de pickle
        prefer="threads"      # ← consistente com o backend
    )(
        delayed(run_dataset_safe)(key) for key in datasets_experiment
    )


    # --- Monta tabelas ---
    classification_rows = []
    regression_rows = []

    for r in all_runs:
        base = {"key": r["key"], "best_model": "", "error": r.get("error", "")}

        if "error" in r and "results" not in r:
            if r.get("task") == "classification":
                classification_rows.append({
                    **base,
                    "accuracy": None,
                    "macro_f1": None,
                    "weighted_f1": None
                })
            else:
                regression_rows.append({
                    **base,
                    "mae": None,
                    "rmse": None,
                    "r2": None
                })
            continue

        if r["task"] == "classification":
            best = max(r["results"], key=lambda m: r["results"][m]["weighted_f1"])
            m = r["results"][best]

            classification_rows.append({
                "key": r["key"],
                "best_model": best,
                "accuracy": round(m["accuracy"], 4),
                "macro_f1": round(m["macro_f1"], 4),
                "weighted_f1": round(m["weighted_f1"], 4),
                "error": "",
            })

        else:
            best = max(r["results"], key=lambda m: r["results"][m]["r2"])
            m = r["results"][best]

            regression_rows.append({
                "key": r["key"],
                "best_model": best,
                "mae": round(m["mae"], 4),
                "rmse": round(m["rmse"], 4),
                "r2": round(m["r2"], 4),
                "error": "",
            })


    # --- salvar ---
    data_path = Path(out_dir)

    if classification_rows:
        clf_df = pd.DataFrame(
            classification_rows,
            columns=["key", "best_model", "accuracy", "macro_f1", "weighted_f1", "error"]
        )
        print("\n=== Classificacao ===")
        display(clf_df)

        clf_path = data_path / "classification_summary.csv"
        clf_df.to_csv(clf_path, index=False)
        print("Salvo em:", clf_path)

    if regression_rows:
        reg_df = pd.DataFrame(
            regression_rows,
            columns=["key", "best_model", "mae", "rmse", "r2", "error"]
        )
        print("\n=== Regressao ===")
        display(reg_df)

        reg_path = data_path / "regression_summary.csv"
        reg_df.to_csv(reg_path, index=False)
        print("Salvo em:", reg_path)

Total cores: 12
Paralelismo externo (datasets): 4


  0%|          | 0/4 [00:00<?, ?it/s]

[cache] data\air_quality_uci\AirQualityUCI.zip
[cache] data\adult\adult.data
[cache] data\communities_crime\communities.data
[cache] data\bank_marketing\bank.zip
[cache] data\air_quality_uci\extracted
[cache] data\adult\adult.test
[cache] data\bank_marketing\extracted
[FAST] Amostra de 3000 linhas usada para air_quality_uci

Dataset: air_quality_uci | Task: regression | Target: C6H6(GT)

[model] ridge_regression
{'mae': 0.8369717204717059, 'rmse': 1.1465453696213777, 'r2': 0.9777383837411985}

Dataset: communities_crime | Task: regression | Target: ViolentCrimesPerPop
[FAST] Amostra de 3000 linhas usada para bank_marketing

Dataset: bank_marketing | Task: classification | Target: y
[FAST] Amostra de 3000 linhas usada para adult

Dataset: adult | Task: classification | Target: income

[model] random_forest
{'mae': 0.025528477248785462, 'rmse': 0.10219051546929972, 'r2': 0.9998231539425029}

------------------------------------------------------------
Rodando: air_quality_uci_MCAR_5

===

 25%|██▌       | 1/4 [00:05<00:16,  5.49s/it]


[model] logistic_regression
{'accuracy': 0.7966666666666666, 'macro_f1': 0.7563768038766707, 'weighted_f1': 0.8095462129683867}

[model] random_forest
{'mae': 0.10123211025731739, 'rmse': 0.14780596061057688, 'r2': 0.5438717754207165}

==== Imputador: knn_3 ====

[model] random_forest
{'accuracy': 0.8566666666666667, 'macro_f1': 0.6831712348953728, 'weighted_f1': 0.861355732390215}

[model] random_forest
{'accuracy': 0.7816666666666666, 'macro_f1': 0.7428944329775506, 'weighted_f1': 0.7964766615092903}

==== Imputador: iterative ====

[model] logistic_regression
{'accuracy': 0.795, 'macro_f1': 0.755563578310739, 'weighted_f1': 0.8082545649661002}


...


[model] random_forest
{'mae': 0.09420806878296861, 'rmse': 0.14117950702646584, 'r2': 0.5838533794728866}

=== Classificacao ===


,key,best_model,accuracy,macro_f1,weighted_f1,error
0,adult,random_forest,0.8200,0.7272,0.8126,
1,bank_marketing,random_forest,0.8733,0.5778,0.8462,


Salvo em: data\classification_summary.csv

=== Regressao ===


,key,best_model,mae,rmse,r2,error
0,air_quality_uci,random_forest,0.0255,0.1022,0.9998,
1,communities_crime,ridge_regression,0.0941,0.1343,0.6232,


Salvo em: data\regression_summary.csv
